# linear-affine-on-custom-tensor — faded example 2: Fill the bias unbroadcast (sum over batch)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `linear-affine-on-custom-tensor`. Running the beacon reports progress on the `Backprop: Linear affine on custom Tensor` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Linear affine on custom Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linear-affine-on-custom-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linear-affine-on-custom-tensor"
DD_SUBTOPIC = "Backprop: Linear affine on custom Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In `out = mm + bias`, the bias broadcasts across the batch axis in the forward. The reverse direction therefore SUMS the upstream `(B, out)` gradient over the batch axis to collapse it back to the bias shape `(out,)`.

## Faded exercise 2

Complete `linear_bias_backward(grad_out, bias)`. It must return the bias gradient of shape `(out,)`. Fill in the reduction that unbroadcasts the batch axis.

**Fill in:** summing grad_out over the batch axis (axis 0) to match the bias shape

In [ ]:
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_bias_backward(grad_out, bias):
    gb = None  # TODO: summing grad_out over the batch axis (axis 0) to match the bias shape
    assert gb.shape == bias.array.shape
    return gb


def _test():
    np.random.seed(8)
    bias = MiniTensor(np.random.randn(4), requires_grad=True)
    grad_out = np.random.randn(6, 4)
    gb = linear_bias_backward(grad_out, bias)
    assert gb.shape == (4,)
    # independent truth: column-wise sum computed by hand
    expected = np.array([sum(grad_out[b, j] for b in range(6)) for j in range(4)])
    assert np.allclose(gb, expected)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
from dataclasses import dataclass

@dataclass
class Recipe:
    func: object
    args: tuple
    kwargs: dict
    parents: dict

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = np.asarray(array, dtype=np.float64)
        self.requires_grad = requires_grad
        self.recipe = recipe

def linear_bias_backward(grad_out, bias):
    gb = grad_out.sum(axis=0)
    assert gb.shape == bias.array.shape
    return gb
```
</details>